In [1]:
!pip install pyspark delta-spark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.1 MB/s eta 0:00:00


In [3]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("LakehousePipeline") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalogo.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

spark

In [6]:
data = [
    (1, "Notebook", 3500),
    (2, "Mouse", 120),
    (3, "Teclado", 250),
    (4, "Monitor", 1200)
]

columns = ["id", "produto", "preco"]

df = spark.createDataFrame(data, columns)

df.show()

+---+--------+-----+
| id| produto|preco|
+---+--------+-----+
|  1|Notebook| 3500|
|  2|   Mouse|  120|
|  3| Teclado|  250|
|  4| Monitor| 1200|
+---+--------+-----+



In [7]:
!wget https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv

--2026-03-11 12:23:30--  https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9729 (9.5K) [text/plain]
Saving to: ‘tips.csv’

tips.csv            100%[===================>]   9.50K  --.-KB/s    in 0s      

2026-03-11 12:23:30 (73.7 MB/s) - ‘tips.csv’ saved [9729/9729]



In [8]:
df = spark.read.csv("tips.csv", header=True, inferSchema=True)
df.show()

+----------+----+------+------+---+------+----+
|total_bill| tip|   sex|smoker|day|  time|size|
+----------+----+------+------+---+------+----+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2|
|     10.27|1.71|  Male|    No|Sun|Dinner|   2|
|     35.26| 5.0|Female|    No|Sun|Dinner|   4|
|     15.42|1.57|  Male|    No|Sun|Dinner|   2|
|     18.43| 3.0|  Male|    No|Sun|Dinner|   4|
|     14.83|3.02|Female|    No|Sun|Dinner|   2|
|     21.58|3.92|  Male|    No|Sun|Dinner|   2|
|     10.33|1.67|Female|    No|Sun|Dinner|   3|
|     16.29|3.71|  Male|    No|Sun|Dinne

In [9]:
df.printSchema()

root
 |-- total_bill: double (nullable = true)
 |-- tip: double (nullable = true)
 |-- sex: string (nullable = true)
 |-- smoker: string (nullable = true)
 |-- day: string (nullable = true)
 |-- time: string (nullable = true)
 |-- size: integer (nullable = true)



In [10]:
df.write.format("delta").mode("overwrite").save("bronze_tips")

In [11]:
bronze_df = spark.read.format("delta").load("bronze_tips")

bronze_df.show()

+----------+----+------+------+---+------+----+
|total_bill| tip|   sex|smoker|day|  time|size|
+----------+----+------+------+---+------+----+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2|
|     10.27|1.71|  Male|    No|Sun|Dinner|   2|
|     35.26| 5.0|Female|    No|Sun|Dinner|   4|
|     15.42|1.57|  Male|    No|Sun|Dinner|   2|
|     18.43| 3.0|  Male|    No|Sun|Dinner|   4|
|     14.83|3.02|Female|    No|Sun|Dinner|   2|
|     21.58|3.92|  Male|    No|Sun|Dinner|   2|
|     10.33|1.67|Female|    No|Sun|Dinner|   3|
|     16.29|3.71|  Male|    No|Sun|Dinne

In [12]:
bronze_df = spark.read.format("delta").load("bronze_tips")


bronze_df.show()

+----------+----+------+------+---+------+----+
|total_bill| tip|   sex|smoker|day|  time|size|
+----------+----+------+------+---+------+----+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2|
|     10.27|1.71|  Male|    No|Sun|Dinner|   2|
|     35.26| 5.0|Female|    No|Sun|Dinner|   4|
|     15.42|1.57|  Male|    No|Sun|Dinner|   2|
|     18.43| 3.0|  Male|    No|Sun|Dinner|   4|
|     14.83|3.02|Female|    No|Sun|Dinner|   2|
|     21.58|3.92|  Male|    No|Sun|Dinner|   2|
|     10.33|1.67|Female|    No|Sun|Dinner|   3|
|     16.29|3.71|  Male|    No|Sun|Dinne

In [13]:
bronze_df.describe().show()


+-------+------------------+------------------+------+------+----+------+------------------+
|summary|        total_bill|               tip|   sex|smoker| day|  time|              size|
+-------+------------------+------------------+------+------+----+------+------------------+
|  count|               244|               244|   244|   244| 244|   244|               244|
|   mean|19.785942622950824|2.9982786885245902|  NULL|  NULL|NULL|  NULL| 2.569672131147541|
| stddev| 8.902411954856857|1.3836381890011815|  NULL|  NULL|NULL|  NULL|0.9510998047322347|
|    min|              3.07|               1.0|Female|    No| Fri|Dinner|                 1|
|    max|             50.81|              10.0|  Male|   Yes|Thur| Lunch|                 6|
+-------+------------------+------------------+------+------+----+------+------------------+



In [14]:
from pyspark.sql.functions import col

silver_df = bronze_df.withColumn(
    "tip_percentage",
    (col("tip")/ col("total_bill")) * 100
)

In [15]:
silver_df.show()

+----------+----+------+------+---+------+----+------------------+
|total_bill| tip|   sex|smoker|day|  time|size|    tip_percentage|
+----------+----+------+------+---+------+----+------------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|5.9446733372572105|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|16.054158607350097|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|16.658733936220845|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2| 13.97804054054054|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|14.680764538430255|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4| 18.62396204033215|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2| 22.80501710376283|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|11.607142857142858|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|13.031914893617023|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2|21.853856562922868|
|     10.27|1.71|  Male|    No|Sun|Dinner|   2| 16.65043816942551|
|     35.26| 5.0|Female|    No|Sun|Dinner|   4|14.180374361883

In [16]:
silver_df = silver_df.filter(col("total_bill") > 5)

In [17]:
silver_df = silver_df.withColumnRenamed("total_bill", "bill_total")

In [18]:
silver_df.write.format("delta").mode("overwrite").save("silver_tips")

In [20]:
silver_table = spark.read.format("delta").load("silver_tips")
silver_table.show()

+----------+----+------+------+---+------+----+------------------+
|bill_total| tip|   sex|smoker|day|  time|size|    tip_percentage|
+----------+----+------+------+---+------+----+------------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|5.9446733372572105|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|16.054158607350097|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|16.658733936220845|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2| 13.97804054054054|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|14.680764538430255|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4| 18.62396204033215|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2| 22.80501710376283|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|11.607142857142858|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|13.031914893617023|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2|21.853856562922868|
|     10.27|1.71|  Male|    No|Sun|Dinner|   2| 16.65043816942551|
|     35.26| 5.0|Female|    No|Sun|Dinner|   4|14.180374361883

In [21]:
silver_df = spark.read.format("delta").load("silver_tips")

silver_df.show()

+----------+----+------+------+---+------+----+------------------+
|bill_total| tip|   sex|smoker|day|  time|size|    tip_percentage|
+----------+----+------+------+---+------+----+------------------+
|     16.99|1.01|Female|    No|Sun|Dinner|   2|5.9446733372572105|
|     10.34|1.66|  Male|    No|Sun|Dinner|   3|16.054158607350097|
|     21.01| 3.5|  Male|    No|Sun|Dinner|   3|16.658733936220845|
|     23.68|3.31|  Male|    No|Sun|Dinner|   2| 13.97804054054054|
|     24.59|3.61|Female|    No|Sun|Dinner|   4|14.680764538430255|
|     25.29|4.71|  Male|    No|Sun|Dinner|   4| 18.62396204033215|
|      8.77| 2.0|  Male|    No|Sun|Dinner|   2| 22.80501710376283|
|     26.88|3.12|  Male|    No|Sun|Dinner|   4|11.607142857142858|
|     15.04|1.96|  Male|    No|Sun|Dinner|   2|13.031914893617023|
|     14.78|3.23|  Male|    No|Sun|Dinner|   2|21.853856562922868|
|     10.27|1.71|  Male|    No|Sun|Dinner|   2| 16.65043816942551|
|     35.26| 5.0|Female|    No|Sun|Dinner|   4|14.180374361883

In [22]:
gold_df = silver_df.groupby("day").agg(
    {"bill_total": "avg", "tip": "avg"}
)

In [23]:
gold_df.show()

+----+------------------+------------------+
| day|          avg(tip)|   avg(bill_total)|
+----+------------------+------------------+
|Thur| 2.771451612903226|17.682741935483865|
| Sun| 3.255131578947369|21.410000000000004|
| Sat|3.0162790697674415| 20.64337209302325|
| Fri| 2.734736842105263|17.151578947368417|
+----+------------------+------------------+



In [24]:
gold_df = gold_df.withColumnRenamed("avg(bill_total)", "avg_bill") \
                .withColumnRenamed("avg(tip)", "avg_tip")

In [25]:
gold_df.write.format("delta").mode("overwrite").save("gold_tips")

In [26]:
gold_table = spark.read.format("delta").load("gold_tips")

gold_table.show()

+----+------------------+------------------+
| day|           avg_tip|          avg_bill|
+----+------------------+------------------+
|Thur| 2.771451612903226|17.682741935483865|
| Sun| 3.255131578947369|21.410000000000004|
| Sat|3.0162790697674415| 20.64337209302325|
| Fri| 2.734736842105263|17.151578947368417|
+----+------------------+------------------+



In [27]:
gold_table.createOrReplaceTempView("tips_gold")

In [28]:
spark.sql("SELECT * FROM tips_gold").show()

+----+------------------+------------------+
| day|           avg_tip|          avg_bill|
+----+------------------+------------------+
|Thur| 2.771451612903226|17.682741935483865|
| Sun| 3.255131578947369|21.410000000000004|
| Sat|3.0162790697674415| 20.64337209302325|
| Fri| 2.734736842105263|17.151578947368417|
+----+------------------+------------------+



In [29]:
spark.sql("""
SELECT day, avg_bill
FROM tips_gold
ORDER BY avg_bill DESC"""). show()

+----+------------------+
| day|          avg_bill|
+----+------------------+
| Sun|21.410000000000004|
| Sat| 20.64337209302325|
|Thur|17.682741935483865|
| Fri|17.151578947368417|
+----+------------------+



In [31]:
spark.sql("""
SELECT
  day,
  avg_bill,
  avg_tip,
  (avg_tip / avg_bill) * 100 AS avg_tip_percentage
FROM tips_gold
ORDER BY avg_tip_percentage DESC
""").show()

+----+------------------+------------------+------------------+
| day|          avg_bill|           avg_tip|avg_tip_percentage|
+----+------------------+------------------+------------------+
| Fri|17.151578947368417| 2.734736842105263| 15.94451945501412|
|Thur|17.682741935483865| 2.771451612903226| 15.67320058741438|
| Sun|21.410000000000004| 3.255131578947369| 15.20379065365422|
| Sat| 20.64337209302325|3.0162790697674415|14.611368027352665|
+----+------------------+------------------+------------------+



In [32]:
gold_pandas = gold_table.toPandas()

In [33]:
gold_pandas.head()

,day,avg_tip,avg_bill
0,Thur,2.771452,17.682742
1,Sun,3.255132,21.410000
2,Sat,3.016279,20.643372
3,Fri,2.734737,17.151579


In [34]:
gold_pandas.to_csv("gold_tips_dashboard.csv", index=False)

In [35]:
from google.colab import files
files.download("gold_tips_dashboard.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [36]:
gold_pandas = gold_table.toPandas()

gold_pandas

,day,avg_tip,avg_bill
0,Thur,2.771452,17.682742
1,Sun,3.255132,21.410000
2,Sat,3.016279,20.643372
3,Fri,2.734737,17.151579


In [37]:
gold_pandas.to_csv(
    "gold_tips_dashboard.csv",
    index=False,
    float_format="%.2f"
)

In [39]:
from google.colab import files
files.download("gold_tips_dashboard.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [40]:
!wget https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv

--2026-03-11 14:50:48--  https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9729 (9.5K) [text/plain]
Saving to: ‘tips.csv.1’

tips.csv.1          100%[===================>]   9.50K  --.-KB/s    in 0s      

2026-03-11 14:50:48 (61.9 MB/s) - ‘tips.csv.1’ saved [9729/9729]



In [41]:
!wget https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv

--2026-03-11 14:50:58--  https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9729 (9.5K) [text/plain]
Saving to: ‘tips.csv.2’

tips.csv.2          100%[===================>]   9.50K  --.-KB/s    in 0s      

2026-03-11 14:50:58 (93.5 MB/s) - ‘tips.csv.2’ saved [9729/9729]



In [42]:
from google.colab import files
files.download("tips.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>